[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/09_causal_attention.ipynb)

# 🔴 Hard: Causal Self-Attention

Implement **causal (masked) self-attention** — the attention used in GPT-style decoders.

Same as softmax attention, but each position can **only attend to itself and earlier positions** (no peeking at future tokens).

$$\text{scores}_{ij} = \begin{cases} \frac{Q_i \cdot K_j}{\sqrt{d_k}} & \text{if } j \le i \\ -\infty & \text{if } j > i \end{cases}$$

### Signature
```python
def causal_attention(Q, K, V):
    # Q, K, V: (batch, seq, d) → output: (batch, seq, d_v)
```

### Rules
- Do **NOT** use `F.scaled_dot_product_attention`
- Position $i$ can only attend to positions $\le i$
- You **may** use `torch.softmax`, `torch.bmm`, `torch.triu`

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.6 MB/s eta 0:00:00


In [2]:
import torch
import math

In [5]:
def causal_attention(Q, K, V):
    """
    Q: (B, S_q, d_q)
    K: (B, S_k, d_k) --> d_k = d_q required
    V: (B, S_k, d_v)

    return: (B, S_q, d_v)
    """

    #general implementation for S_q != S_k (useful for KV-cache when we are running decoding with less than S_k queries)
    d_k = K.shape[-1]
    raw_weights = Q @ K.permute(0, 2, 1) / (d_k ** 0.5)

    total_length = K.shape[1] #S_k
    new_length = Q.shape[1] #S_q
    past_length = total_length - new_length

    q_pos = torch.arange(past_length, total_length).unsqueeze(1) #(new_length,1)
    #e.g. [3, 4, 5] if total = 6 and new = 3
    k_pos = torch.arange(total_length).unsqueeze(0) #(1,total_length)
    # [0, 1, 2, 3, 4, 5]

    # --> each q must give attention only to past keys, not future ones

    mask = k_pos > q_pos #(new_length, total_length) with True where we have to place -inf --> NO ATTTENTION

    raw_weights = raw_weights.masked_fill(mask, float("-inf"))

    att_weights = torch.softmax(raw_weights, dim = -1)

    return att_weights @ V



In [6]:
# 🧪 Debug
torch.manual_seed(0)
Q = torch.randn(1, 4, 8)
K = torch.randn(1, 4, 8)
V = torch.randn(1, 4, 8)
out = causal_attention(Q, K, V)
print("Output shape:", out.shape)          # (1, 4, 8)
print("Pos 0 == V[0]?", torch.allclose(out[:, 0], V[:, 0], atol=1e-5))  # should be True

Output shape: torch.Size([1, 4, 8])
Pos 0 == V[0]? True


In [7]:
from torch_judge import check
check('causal_attention')


🧪 Testing: Causal Self-Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (17.2ms)
  ✅ [2/4] Future tokens don't affect past (8.5ms)
  ✅ [3/4] First position only sees itself (3.6ms)
  ✅ [4/4] Gradient flow (25.2ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (54.5ms total)
  Progress saved. Run status() to see your dashboard.

